<a href="https://colab.research.google.com/github/sara-brxj/MF2143-XRP-Project/blob/main/Copy_of_mediapipe_object_detection_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MediaPipe Object Detection Learning

[![Open In Colab <](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShawnHymel/google-coral-micro-object-detection/blob/master/notebooks/mediapipe-object-detection-learning.ipynb)

```
Original authors: MediaPipeline (Google)
Modified by: Shawn Hymel
Date: December 16, 2023
```

Use transfer learning with Google MediaPipe to build a custom object detection model. Based on the example code from https://developers.google.com/mediapipe/solutions/customization/object_detector.

> **Note:** This script has been verified with TensorFlow v2.15.0.

To use this script, upload your dataset in [Pascal VOC format](http://host.robots.ox.ac.uk/pascal/VOC/) in an archive named *dataset.zip*. You can use a labeling tool like [labelImg](https://github.com/HumanSignal/labelImg) or [Make Sense](https://www.makesense.ai/) to create bounding box annotations in the Pascal VOC format.


Your data should be in the following format. Note that the directory names "Annotations" and "images" must be exactly as shown (with the capital 'A' and lowercase 'i').

```
dataset.zip
├── Annotations/
│   ├── image.01.xml
│   ├── image.02.xml
│   ├── ...
└── images/
    ├── image.01.jpg
    ├── image.02.jpg
    └── ...
```

Run through all the cells. Adjust the hyperparameters (`hparams`) as needed to achieve the desired accuracy. Ideally, you want your average precision (AP) to be greater than 90% to get a useful object detection model.

In [ ]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Configuration

In [ ]:
# Install MediaPipe and Edge TPU compiler
!python --version
!pip install --upgrade pip
!pip install mediapipe-model-maker
! curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
! echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
! sudo apt-get update
! sudo apt-get install edgetpu-compiler

Python 3.11.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 105.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1022  100  1022    0     0   8142      0 --:--:-- --:--:-- --:--:--  8176
OK
deb https://packages.cloud.google.com/apt coral-edgetpu-stable main
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://packages.cloud.google.com/apt coral-edgetpu-stable InRelease [1,423 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://packages.cloud.google.com/apt coral-edgetpu-stable/main amd64 Packages [6,888 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://packages.cloud.google.com/apt coral-edgetpu-stable/main all Packages [1,865 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,204 kB]
G

In [ ]:
from google.colab import files
import os
import json
import tensorflow as tf

from mediapipe_model_maker import object_detector, quantization

/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [ ]:
# Check TensorFlow version
print(tf.__version__)
assert tf.__version__.startswith('2')

2.15.1


In [ ]:
# Settings
BASE_PATH = "."
DATASET_ZIP_PATH = os.path.join(BASE_PATH, "dataset.zip")
DATASET_PATH = os.path.join(BASE_PATH, "dataset/")
TRAIN_SPLIT = 0.8
EXPORT_PATH = os.path.join(BASE_PATH, "exported_models/")
TFLITE_FLOAT32_NAME = "model.tflite"
TFLITE_INT8_NAME = "model_int8.tflite"
METADATA_PATH = os.path.join(EXPORT_PATH, "metadata.json")
METADATA_H_NAME = "metadata.hpp"
METADATA_H_PATH = os.path.join(EXPORT_PATH, METADATA_H_NAME)

## Create dataset

Load and prepare the dataset for training and validation.

In [ ]:
# Unzip dataset
!rm -rf {DATASET_PATH}
!unzip -q {DATASET_ZIP_PATH} -d {DATASET_PATH}

In [ ]:
import os
import shutil

# --- Configuration (Adjust if your folder names change) ---
# The path to the main project directory
PROJECT_ROOT = 'dataset/my-project-name/'
ANNOTATIONS_DIR = os.path.join(PROJECT_ROOT, 'Annotations')
IMAGES_DIR = os.path.join(PROJECT_ROOT, 'images')

# New directory to store unmatched XML files
UNMATCHED_DIR = os.path.join(PROJECT_ROOT, 'unmatched_annotations')

def move_unmatched_annotations(annotations_path, images_path, unmatched_path):
    """
    Moves XML annotation files that do not have a matching JPG image file
    to a separate directory.

    Args:
        annotations_path (str): Path to the directory containing XML files.
        images_path (str): Path to the directory containing JPG files.
        unmatched_path (str): Path to the directory where unmatched XMLs will be moved.
    """
    print(f"Starting cleanup in: {annotations_path}")
    print(f"Checking images in: {images_path}")

    # 1. Create the destination directory if it doesn't exist
    if not os.path.exists(unmatched_path):
        os.makedirs(unmatched_path)
        print(f"Created destination directory: {unmatched_path}\n")
    else:
        print(f"Destination directory already exists: {unmatched_path}\n")

    # 2. Get all XML files
    try:
        xml_files = [f for f in os.listdir(annotations_path) if f.endswith('.xml')]
    except FileNotFoundError:
        print(f"Error: Annotation directory not found at {annotations_path}")
        return

    # 3. Iterate and check for corresponding JPG
    moved_count = 0

    for xml_file in xml_files:
        # Get the base filename without the extension
        base_name = os.path.splitext(xml_file)[0]

        # Construct the expected JPG filename
        expected_jpg_file = base_name + '.jpg'

        # Construct the full path to the expected JPG
        jpg_full_path = os.path.join(images_path, expected_jpg_file)

        # Check if the JPG file exists
        if not os.path.exists(jpg_full_path):
            # If the JPG is missing, move the XML file
            xml_source_path = os.path.join(annotations_path, xml_file)
            xml_destination_path = os.path.join(unmatched_path, xml_file)

            # Print which file is being moved
            print(f"➡️ Moving unmatched annotation: {xml_file} (Missing: {expected_jpg_file})")

            # Move the file using shutil.move
            shutil.move(xml_source_path, xml_destination_path)
            moved_count += 1

    print(f"\n✅ Cleanup complete. Total XML files moved to '{os.path.basename(unmatched_path)}': **{moved_count}**")

# --- Execute the function ---
move_unmatched_annotations(ANNOTATIONS_DIR, IMAGES_DIR, UNMATCHED_DIR)

Starting cleanup in: dataset/my-project-name/Annotations
Checking images in: dataset/my-project-name/images
Created destination directory: dataset/my-project-name/unmatched_annotations

➡️ Moving unmatched annotation: image.c10a949.xml (Missing: image.c10a949.jpg)
➡️ Moving unmatched annotation: image.cb80c807.xml (Missing: image.cb80c807.jpg)
➡️ Moving unmatched annotation: image.c048c402.xml (Missing: image.c048c402.jpg)
➡️ Moving unmatched annotation: image.b72d6cac.xml (Missing: image.b72d6cac.jpg)
➡️ Moving unmatched annotation: image.b7b4be2.xml (Missing: image.b7b4be2.jpg)
➡️ Moving unmatched annotation: image.e55e2d6f.xml (Missing: image.e55e2d6f.jpg)
➡️ Moving unmatched annotation: image.bec07d82.xml (Missing: image.bec07d82.jpg)
➡️ Moving unmatched annotation: image.b085ff1a.xml (Missing: image.b085ff1a.jpg)
➡️ Moving unmatched annotation: image.ad33a619.xml (Missing: image.ad33a619.jpg)
➡️ Moving unmatched annotation: image.ca22da25.xml (Missing: image.ca22da25.jpg)
➡️ Movin

In [ ]:
# Load the dataset
DATASET_PATH="dataset/my-project-name/"
data = object_detector.Dataset.from_pascal_voc_folder(DATASET_PATH)

# Split the dataset into separate training and validation sets
train_data, validation_data = data.split(TRAIN_SPLIT)

In [ ]:
import cv2
import numpy as np
import xml.etree.ElementTree as ET
from glob import glob
import shutil
from tqdm import tqdm


# Augmentation settings
AUGMENTATION_ENABLED = True  # Set to False to skip augmentation
NUM_AUGMENTATIONS = 3        # Number of augmented copies per image


def augment_image(image):
   """Apply random augmentations to an image."""
   augmented = image.copy()

   # Random brightness adjustment (-30 to +30)
   brightness = np.random.randint(-30, 31)
   augmented = np.clip(augmented.astype(np.int16) + brightness, 0, 255).astype(np.uint8)

   # Random contrast adjustment (0.8 to 1.2)
   contrast = np.random.uniform(0.8, 1.2)
   augmented = np.clip(augmented * contrast, 0, 255).astype(np.uint8)

   # Random Gaussian noise (small amount)
   if np.random.random() < 0.3:
       noise = np.random.normal(0, 5, augmented.shape).astype(np.int16)
       augmented = np.clip(augmented.astype(np.int16) + noise, 0, 255).astype(np.uint8)

   # Random slight color shift
   if np.random.random() < 0.3:
       hsv = cv2.cvtColor(augmented, cv2.COLOR_BGR2HSV).astype(np.int16)
       hsv[:, :, 0] = (hsv[:, :, 0] + np.random.randint(-10, 11)) % 180
       hsv[:, :, 1] = np.clip(hsv[:, :, 1] + np.random.randint(-20, 21), 0, 255)
       augmented = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

   return augmented


def augment_dataset(dataset_path, num_augmentations=3):
   """Create augmented copies of all images in the dataset."""
   images_path = os.path.join(dataset_path, "images")
   annotations_path = os.path.join(dataset_path, "Annotations")

   # Get all image files
   image_files = glob(os.path.join(images_path, "*.jpg")) + \
                 glob(os.path.join(images_path, "*.jpeg")) + \
                 glob(os.path.join(images_path, "*.png"))

   original_count = len(image_files)
   augmented_count = 0

   print(f"Found {original_count} original images")
   print(f"Creating {num_augmentations} augmented copies each...")

   for img_path in tqdm(image_files, desc="Augmenting"):
       # Load image
       image = cv2.imread(img_path)
       if image is None:
           continue

       # Get base name and annotation path
       base_name = os.path.splitext(os.path.basename(img_path))[0]
       ext = os.path.splitext(img_path)[1]
       xml_path = os.path.join(annotations_path, f"{base_name}.xml")

       # Skip if no annotation exists
       if not os.path.exists(xml_path):
           continue

       # Create augmented versions
       for i in range(num_augmentations):
           # Augment image
           aug_image = augment_image(image)

           # Save augmented image
           aug_name = f"{base_name}_aug{i}"
           aug_img_path = os.path.join(images_path, f"{aug_name}{ext}")
           cv2.imwrite(aug_img_path, aug_image)

           # Copy and modify annotation
           tree = ET.parse(xml_path)
           root = tree.getroot()

           # Update filename in XML
           filename_elem = root.find('filename')
           if filename_elem is not None:
               filename_elem.text = f"{aug_name}{ext}"

           # Save augmented annotation
           aug_xml_path = os.path.join(annotations_path, f"{aug_name}.xml")
           tree.write(aug_xml_path)

           augmented_count += 1

   print(f"Created {augmented_count} augmented images")
   print(f"Total dataset size: {original_count + augmented_count} images")
   return original_count + augmented_count


# Run augmentation if enabled
if AUGMENTATION_ENABLED:
   total_images = augment_dataset(DATASET_PATH, NUM_AUGMENTATIONS)
else:
   print("Augmentation disabled - using original dataset only")

Found 400 original images
Creating 3 augmented copies each...


Augmenting: 100%|██████████| 400/400 [00:08<00:00, 47.94it/s]

Created 1125 augmented images
Total dataset size: 1525 images


## Train object detection model

Use transfer learning to retrain a model. Gather more/better data and adjust the hyperparameters (`hparams`) to ideally obtain a `total_loss` of less than 0.1 and an average precision (AP) of greater than 0.9.

In [ ]:
# Load pre-trained model and specify hyperparameters
spec = object_detector.SupportedModels.MOBILENET_V2_I320
hparams = object_detector.HParams(
    learning_rate = 0.3,
    batch_size=8,
    epochs=50,
    export_dir=EXPORT_PATH,
)
options = object_detector.ObjectDetectorOptions(
    supported_model=spec,
    hparams=hparams,
)

In [ ]:

model = object_detector.ObjectDetector.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

print("✔ Training started")


/usr/local/lib/python3.11/dist-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


Model: "retina_net_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobile_net (MobileNet)      {'2': (None, 80, 80, 24   2257984   
                             ),                                  
                              '3': (None, 40, 40, 32             
                             ),                                  
                              '4': (None, 20, 20, 96             
                             ),                                  
                              '5': (None, 10, 10, 32             
                             0),                                 
                              '6': (None, 10, 10, 12             
                             80)}                                
                                                                 
 fpn (FPN)                   {'5': (None, 10, 10, 12   130880    
                             8),                  

/usr/local/lib/python3.11/dist-packages/keras/src/backend.py:452: UserWarning: `tf.keras.backend.set_learning_phase` is deprecated and will be removed after 2020-10-11. To update it, simply pass a True/False value to the `training` argument of the `__call__` method of your layer or model.
  warnings.warn(


31/31 [==============================] - 62s 1s/step - total_loss: 2.6551 - cls_loss: 2.2414 - box_loss: 0.0071 - model_loss: 2.5983 - val_total_loss: 1.5933 - val_cls_loss: 1.2378 - val_box_loss: 0.0060 - val_model_loss: 1.5365
Epoch 2/50
31/31 [==============================] - 30s 974ms/step - total_loss: 1.5932 - cls_loss: 1.2355 - box_loss: 0.0060 - model_loss: 1.5364 - val_total_loss: 1.4511 - val_cls_loss: 1.1409 - val_box_loss: 0.0051 - val_model_loss: 1.3943
Epoch 3/50
31/31 [==============================] - 30s 978ms/step - total_loss: 1.3889 - cls_loss: 1.0832 - box_loss: 0.0050 - model_loss: 1.3321 - val_total_loss: 1.2594 - val_cls_loss: 0.9782 - val_box_loss: 0.0045 - val_model_loss: 1.2026
Epoch 4/50
31/31 [==============================] - 30s 971ms/step - total_loss: 1.1291 - cls_loss: 0.8744 - box_loss: 0.0040 - model_loss: 1.0723 - val_total_loss: 0.9581 - val_cls_loss: 0.7171 - val_box_loss: 0.0037 - val_model_loss: 0.9013
Epoch 5/50
31/31 [========================

In [ ]:
# Evaluate model performance
loss, coco_metrics = model.evaluate(
    validation_data,
    batch_size=4,
)
print(f"Validation loss: {loss}")
print(f"Validation metrics: {coco_metrics}")

16/16 [==============================] - 2s 118ms/step - total_loss: 0.4687 - cls_loss: 0.2698 - box_loss: 0.0028 - model_loss: 0.4118
creating index...
index created!
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.22s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.596
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.956
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.675
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.600
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.548
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.582
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.653
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.665
 Average Recall     (AR) @[ 

## Export model

Save the model in three different formats:

 1. 32-bit floating point TensorFlow Lite (TFLite)
 2. 8-bit integer quantized TFLite
 3. TPU compiled and quantized TFLite|

Additionally, save the metadata (anchor box information) in a .h file that a resource-constrained device can recalculate the anchor boxes.



In [ ]:
# Export 32-bit float model
model.export_model()

Exporting a floating point model


/usr/local/lib/python3.11/dist-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


In [ ]:
# Perform post-training quantization (8-bit integer) and save quantized model
quantization_config = quantization.QuantizationConfig.for_int8(
    representative_data=validation_data,
)
model.restore_float_ckpt()
model.export_model(
    model_name=TFLITE_INT8_NAME,
    quantization_config=quantization_config,
)

/usr/local/lib/python3.11/dist-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


Using existing files at /tmp/model_maker/object_detector/mobilenetv2_i320
Model: "retina_net_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobile_net_1 (MobileNet)    {'2': (None, 80, 80, 24   2257984   
                             ),                                  
                              '3': (None, 40, 40, 32             
                             ),                                  
                              '4': (None, 20, 20, 96             
                             ),                                  
                              '5': (None, 10, 10, 32             
                             0),                                 
                              '6': (None, 10, 10, 12             
                             80)}                                
                                                                 
 fpn_1 (FPN)                 {'5': (None

/usr/local/lib/python3.11/dist-packages/keras/src/engine/functional.py:642: UserWarning: Input dict contained keys ['6'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)


In [ ]:
!sudo apt-get update
!sudo apt-get install -y edgetpu-compiler

# Compile the model
import os
EXPORT_PATH = "/content/exported_models"           # your folder path
TFLITE_INT8_NAME = "model_int8.tflite"    # your model name

tflite_path = os.path.join(EXPORT_PATH, TFLITE_INT8_NAME)
!edgetpu_compiler -s -o {EXPORT_PATH} {tflite_path}

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://packages.cloud.google.com/apt coral-edgetpu-stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: https://packages.cloud.google.com/apt/dists/coral-edgetpu-stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-k

In [ ]:
# Import model metadata
with open(METADATA_PATH, 'r') as file:
    metadata = json.load(file)

# Parse metadata
custom_metadata = metadata['subgraph_metadata'][0]['custom_metadata'][0]
anchors = custom_metadata['data']['ssd_anchors_options']['fixed_anchors_schema']['anchors']
num_values_per_keypoint = custom_metadata['data']['tensors_decoding_options']['num_values_per_keypoint']
apply_exponential_on_box_size = custom_metadata['data']['tensors_decoding_options']['apply_exponential_on_box_size']
x_scale = custom_metadata['data']['tensors_decoding_options']['x_scale']
y_scale = custom_metadata['data']['tensors_decoding_options']['y_scale']
w_scale = custom_metadata['data']['tensors_decoding_options']['w_scale']
h_scale = custom_metadata['data']['tensors_decoding_options']['h_scale']

In [ ]:
# Figure out when the resets (sectors) occur, the x/y increases, and width/height of anchors
reset_idxs = []
y_strides = []
x_strides = []
widths_per_section = []
widths = []
heights_per_section = []
heights = []
reset_flag = True
x_stride_flag = True
width_flag = True

# Go through all the anchors
num_anchors = len(anchors)
for i in range(num_anchors):

    # Store the first index
    if i == 0:
        reset_idxs.append(i)

    # Only measure strides on not 0 indexes
    else:

        # New section: reset flags
        if anchors[i]['y_center'] < anchors[i - 1]['y_center']:
            reset_idxs.append(i)
            reset_flag = True
            x_stride_flag = True
            width_flag = True

        # Measure Y increase (stride)
        if reset_flag:
            if anchors[i]['y_center'] > anchors[i - 1]['y_center']:
                y_inc = anchors[i]['y_center'] - anchors[i - 1]['y_center']
                y_strides.append(round(y_inc, 5))
                reset_flag = False

        # Measure X increase (stride)
        if x_stride_flag:
            if anchors[i]['x_center'] > anchors[i - 1]['x_center']:
                x_inc = anchors[i]['x_center'] - anchors[i - 1]['x_center']
                x_strides.append(round(x_inc, 5))
                x_stride_flag = False

    # Record widths and heights of the anchor boxes
    if width_flag:
        if i != 0 and anchors[i]['x_center'] > anchors[i - 1]['x_center']:
            widths.append(widths_per_section)
            widths_per_section = []
            heights.append(heights_per_section)
            heights_per_section = []
            width_flag = False
        else:
            width = anchors[i]['width']
            widths_per_section.append(round(width, 5))
            height = anchors[i]['height']
            heights_per_section.append(round(height, 5))

# Calculate the number of sectors
num_sectors = len(reset_idxs)

# Calculate the number of anchors per coordinate
num_anchors_per_coord = len(widths[0])

# Calculate the number of Xs in each Y
num_xs_per_y = []
for sector in range(num_sectors):
    num_xs_per_y.append(int(1.0 / x_strides[sector] * num_anchors_per_coord))

print(f"Number of anchors {num_anchors}")
print(f"Number of sectors: {num_sectors}")
print(f"Number of anchors per coordinate: {num_anchors_per_coord}")
print(f"Reset indexes: {reset_idxs}")
print(f"Number of Xs per Y: {num_xs_per_y}")
print(f"X strides: {x_strides}")
print(f"Y strides: {y_strides}")
print("Widths:")
for wps in widths:
    print(wps)
print("Heights:")
for hps in heights:
    print(hps)

Number of anchors 19125
Number of sectors: 4
Number of anchors per coordinate: 9
Reset indexes: [0, 14400, 18000, 18900]
Number of Xs per Y: [360, 180, 90, 45]
X strides: [0.025, 0.05, 0.1, 0.2]
Y strides: [0.025, 0.05, 0.1, 0.2]
Widths:
[0.05303, 0.075, 0.10607, 0.06682, 0.09449, 0.13364, 0.08418, 0.11905, 0.16837]
[0.10607, 0.15, 0.21213, 0.13364, 0.18899, 0.26727, 0.16837, 0.23811, 0.33674]
[0.21213, 0.3, 0.42426, 0.26727, 0.37798, 0.53454, 0.33674, 0.47622, 0.67348]
[0.42426, 0.6, 0.84853, 0.53454, 0.75595, 1.06908, 0.67348, 0.95244, 1.34695]
Heights:
[0.10607, 0.075, 0.05303, 0.13364, 0.09449, 0.06682, 0.16837, 0.11905, 0.08418]
[0.21213, 0.15, 0.10607, 0.26727, 0.18899, 0.13364, 0.33674, 0.23811, 0.16837]
[0.42426, 0.3, 0.21213, 0.53454, 0.37798, 0.26727, 0.67348, 0.47622, 0.33674]
[0.84853, 0.6, 0.42426, 1.06908, 0.75595, 0.53454, 1.34695, 0.95244, 0.67348]


In [ ]:
# Generate header file for metadata information
h_str = f"""\
// Filename: {METADATA_H_NAME}

#ifndef METADATA_HPP
#define METADATA_HPP

namespace metadata {{
    constexpr unsigned int num_anchors = {num_anchors};
    constexpr int apply_exp_scaling = {1 if apply_exponential_on_box_size else 0};
    constexpr float x_scale = {x_scale};
    constexpr float y_scale = {y_scale};
    constexpr float w_scale = {w_scale};
    constexpr float h_scale = {h_scale};
    constexpr unsigned int num_sectors = {num_sectors};
    constexpr unsigned int num_anchors_per_coord = {num_anchors_per_coord};
"""

# Print reset indexes
h_str += "    constexpr unsigned int reset_idxs[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{reset_idxs[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the number of X values for each Y value
h_str += "    constexpr unsigned int num_xs_per_y[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{num_xs_per_y[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the X strides
h_str += "    constexpr float x_strides[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{x_strides[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the Y strides
h_str += "    constexpr float y_strides[] = {\r\n"
h_str += "        "
for i in range(num_sectors):
    h_str += f"{y_strides[i]}"
    if i < num_sectors - 1:
        h_str += ", "
h_str += "\r\n"
h_str += "    };\r\n"

# Print the anchor widths for each section
h_str += f"    constexpr float widths[{num_sectors}][{len(widths[0])}] = {{\r\n"
for i in range(num_sectors):
    h_str += "        {"
    for j in range(len(widths[0])):
        h_str += f"{widths[i][j]}"
        if j < len(widths[0]) - 1:
            h_str += ", "
    h_str += "}"
    if i < num_sectors - 1:
        h_str += ","
    h_str += "\r\n"
h_str += "    };\r\n"

# Print the anchor heights for each section
h_str += f"    constexpr float heights[{num_sectors}][{len(heights[0])}] = {{\r\n"
for i in range(num_sectors):
    h_str += "        {"
    for j in range(len(heights[0])):
        h_str += f"{heights[i][j]}"
        if j < len(heights[0]) - 1:
            h_str += ", "
    h_str += "}"
    if i < num_sectors - 1:
        h_str += ","
    h_str += "\r\n"
h_str += "    };\r\n"

# Close header file
h_str += """\
}

#endif // METADATA_HPP
"""

# write to .h file
with open(METADATA_H_PATH, 'w') as file:
    file.write(h_str)

In [ ]:
# Zip exported models
zip_name = os.path.normpath(EXPORT_PATH).split(os.sep)[-1] + ".zip"
zip_path = os.path.join(BASE_PATH, zip_name)
!zip -q -r {zip_path} {EXPORT_PATH}/*

In [ ]:
# Download exported models
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -q -r exported_models.zip {os.path.join(EXPORT_PATH, "*")}